In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
#Check GPU enabled
import torch
import pandas   as pd
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from pathlib import Path
import shutil
print(torch.cuda.is_available())    

True


In [2]:
# Load the sequences and labels for training
loaded_sequences_df = pd.read_pickle('/content/drive/MyDrive/capstone_team_3/data_set/classic_midi_sequences_256.pkl')
print("MIDI sequences loaded from pickle file.")
print(f"Shape of loaded sequences DataFrame: {loaded_sequences_df.shape}")  

MIDI sequences loaded from pickle file.
Shape of loaded sequences DataFrame: (630987, 2)


In [3]:
train_df, test_df = train_test_split(loaded_sequences_df, test_size=0.2, random_state=42)
print(f"Training set size: {len(train_df)}")
print(f"Testing set size: {len(test_df)}")  

Training set size: 504789
Testing set size: 126198


In [4]:
def cosine_similarity(vec1, vec2):
    vec1 = np.asarray(vec1, dtype=np.float32)
    vec2 = np.asarray(vec2, dtype=np.float32)
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0
    return float(dot_product / (norm_vec1 * norm_vec2))

def features_to_vector(features):
    features = np.asarray(features, dtype=np.float32)
    if features.ndim == 1:
        features = features.reshape(1, -1)
    summary_vector = np.concatenate([
        features.mean(axis=0),
        features.std(axis=0),
        features.min(axis=0),
        features.max(axis=0)
    ])
    return summary_vector.astype(np.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


### GRU  model 
Prepare normalized sequence tensors, train a simple LSTM classifier, and compare learned embeddings with cosine similarity.

In [5]:
# Prepare normalized tensors and dataloaders
train_sequences = np.stack(train_df['sequence'].to_numpy()).astype(np.float32)
test_sequences = np.stack(test_df['sequence'].to_numpy()).astype(np.float32)

feature_mean = train_sequences.reshape(-1, train_sequences.shape[-1]).mean(axis=0)
feature_std = train_sequences.reshape(-1, train_sequences.shape[-1]).std(axis=0) + 1e-6

# Normalize the sequences
X_train = (train_sequences - feature_mean) / feature_std
X_test = (test_sequences - feature_mean) / feature_std

label_encoder = LabelEncoder()
label_encoder.fit(loaded_sequences_df['label'])
y_train = label_encoder.transform(train_df['label'])
y_test = label_encoder.transform(test_df['label'])

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

batch_size = 64
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=batch_size, shuffle=False)

input_dim = X_train.shape[-1]
sequence_length = X_train.shape[1]
num_classes = len(label_encoder.classes_)

print(f'Train tensor shape: {X_train_tensor.shape}')
print(f'Test tensor shape: {X_test_tensor.shape}')
print(f'Input dimension: {input_dim}, Sequence length: {sequence_length}, Classes: {num_classes}')

Train tensor shape: torch.Size([504789, 256, 3])
Test tensor shape: torch.Size([126198, 256, 3])
Input dimension: 3, Sequence length: 256, Classes: 289


In [22]:
 #Define and train a GRU sequence classifier
class GRUSequenceClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, return_embedding=False):
        _, hidden_state = self.gru(x)
        embedding = self.dropout(hidden_state[-1])
        logits = self.classifier(embedding)
        if return_embedding:
            return logits, embedding
        return logits


In [7]:
# Hyperparameters
gru_hidden_dim = 128
gru_learning_rate = 1e-3
gru_epochs = 5


In [23]:
# Initialize model, loss function, and optimizer
gru_model = GRUSequenceClassifier(
    input_dim=input_dim,
    hidden_dim=gru_hidden_dim,
    num_classes=num_classes
).to(device)

gru_criterion = nn.CrossEntropyLoss()
gru_optimizer = optim.Adam(gru_model.parameters(), lr=gru_learning_rate)


In [9]:
gru_history = []
for epoch in range(gru_epochs):
    gru_model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_inputs, batch_labels in train_loader:
        batch_inputs = batch_inputs.to(device)
        batch_labels = batch_labels.to(device)

        gru_optimizer.zero_grad()
        logits = gru_model(batch_inputs)
        loss = gru_criterion(logits, batch_labels)
        loss.backward()
        gru_optimizer.step()

        train_loss += loss.item() * batch_inputs.size(0)
        train_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
        train_total += batch_labels.size(0)

    gru_model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for batch_inputs, batch_labels in test_loader:
            batch_inputs = batch_inputs.to(device)
            batch_labels = batch_labels.to(device)

            logits = gru_model(batch_inputs)
            loss = gru_criterion(logits, batch_labels)

            preds = logits.argmax(dim=1)
            test_loss += loss.item() * batch_inputs.size(0)
            test_correct += (preds == batch_labels).sum().item()
            test_total += batch_labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(batch_labels.cpu().numpy())

    epoch_metrics = {
        'epoch': epoch + 1,
        'train_loss': train_loss / train_total,
        'train_accuracy': train_correct / train_total,
        'test_loss': test_loss / test_total,
        'test_accuracy': test_correct / test_total,
        'test_accuracy_sklearn': accuracy_score(all_targets, all_preds)
    }
    gru_history.append(epoch_metrics)
    print(
        f"GRU Epoch {epoch + 1}/{gru_epochs} - "
        f"train_loss: {epoch_metrics['train_loss']:.4f}, "
        f"train_accuracy: {epoch_metrics['train_accuracy']:.4f}, "
        f"test_loss: {epoch_metrics['test_loss']:.4f}, "
        f"test_accuracy: {epoch_metrics['test_accuracy']:.4f}"
    )

gru_history_df = pd.DataFrame(gru_history)
print('\nGRU training complete.')
gru_history_df

GRU Epoch 1/5 - train_loss: 2.0995, train_accuracy: 0.4955, test_loss: 0.3764, test_accuracy: 0.9186
GRU Epoch 2/5 - train_loss: 0.3139, train_accuracy: 0.9151, test_loss: 0.0500, test_accuracy: 0.9899
GRU Epoch 3/5 - train_loss: 0.1097, train_accuracy: 0.9697, test_loss: 0.0137, test_accuracy: 0.9976
GRU Epoch 4/5 - train_loss: 0.0653, train_accuracy: 0.9819, test_loss: 0.0165, test_accuracy: 0.9960
GRU Epoch 5/5 - train_loss: 0.0456, train_accuracy: 0.9872, test_loss: 0.0034, test_accuracy: 0.9994

GRU training complete.


,epoch,train_loss,train_accuracy,test_loss,test_accuracy,test_accuracy_sklearn
0,1,2.099477,0.495542,0.376423,0.918644,0.918644
1,2,0.313876,0.915139,0.049970,0.989881,0.989881
2,3,0.109714,0.969676,0.013708,0.997567,0.997567
3,4,0.065331,0.981913,0.016522,0.995975,0.995975
4,5,0.045583,0.987232,0.003402,0.999358,0.999358


In [10]:
# Evaluate GRU predictions on the test set and compare with LSTM results
gru_model.eval()
gru_predictions = []
with torch.no_grad():
    for batch_inputs, batch_labels in test_loader:
        batch_inputs = batch_inputs.to(device)
        batch_labels = batch_labels.to(device)
        logits = gru_model(batch_inputs)
        predictions = logits.argmax(dim=1)
        gru_predictions.extend(predictions.cpu().numpy())   

In [11]:
# Test set accuracy for GRU
gru_test_accuracy = accuracy_score(all_targets, gru_predictions)
print(f'GRU Test Accuracy: {gru_test_accuracy:.4f}')

GRU Test Accuracy: 0.9994


In [12]:
# Print sample test sequences with predicted labels for GRU
test_df['predicted_label'] = gru_predictions
print("Sample test sequences with predicted labels (GRU):")
print(test_df.head())

Sample test sequences with predicted labels (GRU):
                                                 sequence  \
305937  [[0.10387733541666933, 1.0, 56.0], [0.10412864...   
18265   [[0.181796250000005, 6.0, 44.0], [0.3635924999...   
440965  [[0.0625, 16.0, 32.0], [0.031645562499988955, ...   
120673  [[0.182774749999993, -4.0, 42.0], [0.194715250...   
437541  [[0.3313694999999939, 20.0, 83.0], [0.33136949...   

                       label  predicted_label  
305937  burg_spinnerlied.mid               58  
18265            bor_ps7.mid               42  
440965     liz_et_trans4.mid              164  
120673          appass_1.mid               14  
437541     liz_et_trans4.mid              164  


In [14]:
# Store the GRU model state for later use
gru_model_save_path = '/content/drive/MyDrive/capstone_team_3/models/gru_sequence_classifier.pth'
torch.save(gru_model.state_dict(), gru_model_save_path)
print(f"GRU model state saved to {gru_model_save_path}")

GRU model state saved to /content/drive/MyDrive/capstone_team_3/models/gru_sequence_classifier.pth


In [24]:
#load the model state for evaluation on external test set
gru_model.load_state_dict(torch.load(gru_model_save_path))
gru_model.to(device)
gru_model.eval()

GRUSequenceClassifier(
  (gru): GRU(3, 128, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=128, out_features=289, bias=True)
)

In [25]:
# Compute GRU prototype embeddings (mean embedding per class from training set)
gru_model.eval()
gru_embedding_sums = np.zeros((num_classes, gru_hidden_dim), dtype=np.float32)
gru_embedding_counts = np.zeros(num_classes, dtype=np.int32)

with torch.no_grad():
    for batch_inputs, batch_labels in train_loader:
        batch_inputs = batch_inputs.to(device)
        batch_labels = batch_labels.to(device)
        _, embeddings = gru_model(batch_inputs, return_embedding=True)
        embeddings = embeddings.cpu().numpy()
        labels_np = batch_labels.cpu().numpy()
        for emb, lbl in zip(embeddings, labels_np):
            gru_embedding_sums[lbl] += emb
            gru_embedding_counts[lbl] += 1

gru_prototype_embeddings = gru_embedding_sums / np.maximum(gru_embedding_counts[:, None], 1)

# Run inference on a sample sequence
sample_index = 0
sample_sequence = X_test_tensor[sample_index:sample_index + 1].to(device)
sample_true_label = label_encoder.inverse_transform([y_test[sample_index]])[0]

with torch.no_grad():
    sample_logits, sample_embedding = gru_model(sample_sequence, return_embedding=True)
    sample_prediction = int(sample_logits.argmax(dim=1).item())
    sample_embedding = sample_embedding.squeeze(0).cpu().numpy()

print(f'Sample true label: {sample_true_label}')
print(f"Sample predicted label: {label_encoder.inverse_transform([sample_prediction])[0]}")
print(f"Sample embedding shape: {sample_embedding.shape}")

similarity_rows = []
for class_index, class_name in enumerate(label_encoder.classes_):
    similarity_rows.append({
        'label': class_name,
        'cosine_similarity': cosine_similarity(sample_embedding, gru_prototype_embeddings[class_index])
    })


Sample true label: burg_spinnerlied.mid
Sample predicted label: burg_spinnerlied.mid
Sample embedding shape: (128,)


In [26]:
similarity_df = pd.DataFrame(similarity_rows).sort_values('cosine_similarity', ascending=False).head(5)
print(f'Sample true label: {sample_true_label}')
print(f"Sample predicted label: {label_encoder.inverse_transform([sample_prediction])[0]}")
similarity_df

Sample true label: burg_spinnerlied.mid
Sample predicted label: burg_spinnerlied.mid


,label,cosine_similarity
58,burg_spinnerlied.mid,0.730782
277,ty_februar.mid,0.566800
56,burg_perlen.mid,0.566284
270,scn16_7.mid,0.523317
89,chpn_op23.mid,0.519797


In [27]:
# Load the sequences and labels for testing on external MIDI files
external_test_df = pd.read_pickle('/content/drive/MyDrive/capstone_team_3/data_set/external_test_sequences.pkl')
print("External MIDI sequences loaded from pickle file.")
print(f"Shape of external test sequences DataFrame: {external_test_df.shape}")
print("Sample external test sequences:")
external_test_df.head()

External MIDI sequences loaded from pickle file.
Shape of external test sequences DataFrame: (203723, 2)
Sample external test sequences:


,sequence,label
0,"[[0.7999999999999998, 0.0, 44.0], [0.300000000...",Piano Sonata n12 K332.mid
1,"[[0.30000000000000027, 4.0, 47.0], [0.79999999...",Piano Sonata n12 K332.mid
2,"[[0.7999999999999998, 3.0, 49.0], [0.304166666...",Piano Sonata n12 K332.mid
3,"[[0.3041666666666667, -3.0, 52.0], [0.79999999...",Piano Sonata n12 K332.mid
4,"[[0.7999999999999998, 1.0, 54.0], [0.299999999...",Piano Sonata n12 K332.mid


In [28]:
X_test_external= np.stack(external_test_df['sequence'].to_numpy()).astype(np.float32)
X_test_external = (X_test_external - feature_mean) / feature_std

In [30]:
# Test with the merged test sequences and predicted labels from the GRU model
import tqdm
gru_test_predictions = []

with torch.no_grad():
    for seq in tqdm.tqdm(X_test_external):
        seq_tensor = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
        logits = gru_model(seq_tensor)
        predicted_label_index = int(logits.argmax(dim=1).item())
        gru_test_predictions.append(label_encoder.inverse_transform([predicted_label_index])[0])

100%|██████████| 203723/203723 [05:25<00:00, 625.72it/s]


In [33]:
# Add predictions to  Xtest_df
external_test_df['gru_predicted_label'] = gru_test_predictions
print("Sample test sequences with GRU predicted labels:")
external_test_df.head()

Sample test sequences with GRU predicted labels:


,sequence,label,gru_predicted_label
0,"[[0.7999999999999998, 0.0, 44.0], [0.300000000...",Piano Sonata n12 K332.mid,mz_333_1.mid
1,"[[0.30000000000000027, 4.0, 47.0], [0.79999999...",Piano Sonata n12 K332.mid,mz_333_1.mid
2,"[[0.7999999999999998, 3.0, 49.0], [0.304166666...",Piano Sonata n12 K332.mid,mz_333_1.mid
3,"[[0.3041666666666667, -3.0, 52.0], [0.79999999...",Piano Sonata n12 K332.mid,mz_333_1.mid
4,"[[0.7999999999999998, 1.0, 54.0], [0.299999999...",Piano Sonata n12 K332.mid,mz_333_1.mid


In [38]:
# Stoe the  results in a pickle file for later analysis (only labels and predicted labels)
external_test_df[['label', 'gru_predicted_label']].to_pickle('/content/drive/MyDrive/capstone_team_3/results/external_test_predictions_gru_v1.pkl')
print("External test predictions saved to pickle file.")

External test predictions saved to pickle file.


In [39]:
# Store model details as meta for the completed run
final_epoch = gru_history_df.iloc[-1]

model_meta_data = {
    "modelName": "GRUSequenceClassifier",
    "checkpoint": gru_model_save_path,
    "hyperParams": {
        "hidden_dim": gru_hidden_dim,
        "num_layers": 2,
        "dropout": 0.3,
        "learning_rate": gru_learning_rate,
        "epochs": gru_epochs,
        "batch_size": batch_size,
        "optimizer": "Adam",
        "loss": "CrossEntropyLoss",
    },
    "dataParams": {
        "input_dim": input_dim,
        "sequence_length": sequence_length,
        "num_classes": num_classes,
        "train_size": len(train_df),
        "test_size": len(test_df),
    },
    "results": {
        "final_train_loss": round(float(final_epoch['train_loss']), 4),
        "final_train_accuracy": round(float(final_epoch['train_accuracy']), 4),
        "final_test_loss": round(float(final_epoch['test_loss']), 4),
        "final_test_accuracy": round(float(final_epoch['test_accuracy']), 4),
        "test_accuracy_sklearn": round(float(gru_test_accuracy), 4),
    },
    "trainingHistory": gru_history_df.to_dict(orient='records'),
}

import json
print(json.dumps({k: v for k, v in model_meta_data.items() if k != 'trainingHistory'}, indent=2))
print(f"\nTraining history ({len(model_meta_data['trainingHistory'])} epochs) stored in model_meta_data['trainingHistory']")


{
  "modelName": "GRUSequenceClassifier",
  "checkpoint": "/content/drive/MyDrive/capstone_team_3/models/gru_sequence_classifier.pth",
  "hyperParams": {
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.3,
    "learning_rate": 0.001,
    "epochs": 5,
    "batch_size": 64,
    "optimizer": "Adam",
    "loss": "CrossEntropyLoss"
  },
  "dataParams": {
    "input_dim": 3,
    "sequence_length": 256,
    "num_classes": 289,
    "train_size": 504789,
    "test_size": 126198
  },
  "results": {
    "final_train_loss": 0.0456,
    "final_train_accuracy": 0.9872,
    "final_test_loss": 0.0034,
    "final_test_accuracy": 0.9994,
    "test_accuracy_sklearn": 0.9994
  }
}

Training history (5 epochs) stored in model_meta_data['trainingHistory']


In [40]:
# Store json file in models in google drive
import json

meta_save_path = '/content/drive/MyDrive/capstone_team_3/models/gru_sequence_classifier_meta.json'
with open(meta_save_path, 'w') as f:
    json.dump(model_meta_data, f, indent=2)

print(f"Model metadata saved to {meta_save_path}")


Model metadata saved to /content/drive/MyDrive/capstone_team_3/models/gru_sequence_classifier_meta.json
